# ABPNoiseSweep plotting notebook

In [45]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## 1. Choose the run folder

Change `RUN_DIR` to the folder where the cluster run stored the data.

Examples:

```python
RUN_DIR = Path(r"C:\Users\user\Desktop\Project\data\runs\run_name")
RUN_DIR = Path("/scratch/myuser/ABP_runs/run_name")
RUN_DIR = Path("./abp_noise_sweep_endpoint_conditioned")
```

In [46]:

RUN_DIR = Path(r"C:\Users\amarin\Desktop\ABP_Rare_Events\ABPNoiseSweep\abp_noise_sweep_smoke_test")

DATA_DIR = RUN_DIR / "data"
PLOTS_DIR = RUN_DIR / "plots"

SAVE_FIGS = True
SHOW_FIGS = True
FIG_DPI = 180

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_DIR   = {RUN_DIR.resolve()}")
print(f"DATA_DIR  = {DATA_DIR.resolve()}")
print(f"PLOTS_DIR = {PLOTS_DIR.resolve()}")
print(f"DATA_DIR exists? {DATA_DIR.exists()}")

RUN_DIR   = C:\Users\amarin\Desktop\ABP_Rare_Events\ABPNoiseSweep\abp_noise_sweep_smoke_test
DATA_DIR  = C:\Users\amarin\Desktop\ABP_Rare_Events\ABPNoiseSweep\abp_noise_sweep_smoke_test\data
PLOTS_DIR = C:\Users\amarin\Desktop\ABP_Rare_Events\ABPNoiseSweep\abp_noise_sweep_smoke_test\plots
DATA_DIR exists? False


## 2. Helper functions

These helpers load the CSV files, find all per-noise folders, and save each figure into `RUN_DIR/plots`.

In [47]:
FIGURES = []


def read_csv_if_exists(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing, skipped: {path}")
        return None
    return pd.read_csv(path, **kwargs)


def read_key_value_csv(path):
    df = read_csv_if_exists(path)
    if df is None or not {"key", "value"}.issubset(df.columns):
        return {}
    out = {}
    for _, row in df.iterrows():
        key = str(row["key"])
        value = row["value"]
        numeric_value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
        if pd.notna(numeric_value):
            out[key] = numeric_value
        else:
            value_str = str(value)
            if value_str.lower() == "true":
                out[key] = True
            elif value_str.lower() == "false":
                out[key] = False
            else:
                out[key] = value
    return out


def find_case_dirs(data_dir=DATA_DIR):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        return []
    dirs = [p for p in data_dir.iterdir() if p.is_dir() and (p / "case_summary.csv").exists()]
    def d_value(p):
        summary = read_key_value_csv(p / "case_summary.csv")
        return float(summary.get("D", np.nan))
    return sorted(dirs, key=d_value, reverse=True)


def finite_df(df, columns):
    out = df.copy()
    for col in columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    mask = np.ones(len(out), dtype=bool)
    for col in columns:
        mask &= np.isfinite(out[col])
    return out.loc[mask].copy()


def safe_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")


def save_and_maybe_show(fig, filename):
    path = PLOTS_DIR / filename
    if SAVE_FIGS:
        fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
        FIGURES.append(path)
    if SHOW_FIGS:
        plt.show()
    else:
        plt.close(fig)
    return path


def label_for_case(case_dir):
    summary = read_key_value_csv(case_dir / "case_summary.csv")
    D = summary.get("D", case_dir.name)
    try:
        return f"D={float(D):g}"
    except Exception:
        return str(D)


def case_tag(case_dir):
    return safe_name(case_dir.name)


def load_transition_ensemble(case_dir):
    traj = read_csv_if_exists(case_dir / "endpoint_x_gt_0p5_trajectory_summary.csv")
    path = read_csv_if_exists(case_dir / "endpoint_x_gt_0p5_filtered_path_xy_long.csv")
    return traj, path


def restrict_range(df, col, value_range):
    """Return only rows with col inside value_range = (min, max)."""
    if value_range is None:
        return df
    lo, hi = value_range
    work = df.copy()
    work[col] = pd.to_numeric(work[col], errors="coerce")
    if lo is not None:
        work = work[work[col] >= lo]
    if hi is not None:
        work = work[work[col] <= hi]
    return work


def plot_weighted_heatmap(df, xcol, ycol, weight_col, title, filename, x_range=None, y_range=None, bins=80, log_scale=True, cmap="magma"):
    needed = {xcol, ycol, weight_col}
    if not needed.issubset(df.columns):
        print(f"Missing columns for {filename}: needed {needed}")
        return None

    work = df[[xcol, ycol, weight_col]].copy()
    for col in [xcol, ycol, weight_col]:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    work = work.dropna()
    work = restrict_range(work, xcol, x_range)
    work = restrict_range(work, ycol, y_range)
    if work.empty:
        print(f"No numeric data for {filename} inside x_range={x_range}, y_range={y_range}")
        return None

    x_edges = np.linspace(x_range[0] if x_range and x_range[0] is not None else work[xcol].min(),
                          x_range[1] if x_range and x_range[1] is not None else work[xcol].max(), bins + 1)
    y_edges = np.linspace(y_range[0] if y_range and y_range[0] is not None else work[ycol].min(),
                          y_range[1] if y_range and y_range[1] is not None else work[ycol].max(), bins + 1)
    hist, x_edges, y_edges = np.histogram2d(work[xcol], work[ycol], bins=[x_edges, y_edges], weights=work[weight_col])

    fig, ax = plt.subplots(figsize=(6.8, 5.2))
    positive = hist[np.isfinite(hist) & (hist > 0)]
    if log_scale and positive.size > 0:
        from matplotlib.colors import LogNorm
        mesh = ax.pcolormesh(x_edges, y_edges, hist.T, shading="auto", cmap=cmap, norm=LogNorm(vmin=positive.min(), vmax=positive.max()))
    else:
        mesh = ax.pcolormesh(x_edges, y_edges, hist.T, shading="auto", cmap=cmap)
    fig.colorbar(mesh, ax=ax, label=weight_col)
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_title(title)
    ax.grid(False)
    return save_and_maybe_show(fig, filename)



def fit_transition_law(summary_df):
    needed = {"D", "endpoint_x_gt_0p5_transition_rate"}
    if not needed.issubset(summary_df.columns):
        return None
    work = summary_df.copy()
    work["D"] = pd.to_numeric(work["D"], errors="coerce")
    work["endpoint_x_gt_0p5_transition_rate"] = pd.to_numeric(work["endpoint_x_gt_0p5_transition_rate"], errors="coerce")
    work = work[np.isfinite(work["D"]) & np.isfinite(work["endpoint_x_gt_0p5_transition_rate"]) & (work["D"] > 0) & (work["endpoint_x_gt_0p5_transition_rate"] > 0)]
    if len(work) < 2:
        return None
    invD = 1.0 / work["D"].to_numpy(dtype=float)
    log_r = np.log(work["endpoint_x_gt_0p5_transition_rate"].to_numpy(dtype=float))
    barrier, logA = np.polyfit(invD, log_r, 1)
    return {
        "Barrier": float(barrier),
        "A": float(np.exp(logA)),
        "r_fit": lambda D: float(np.exp(logA + barrier / D)),
        "n_points": int(len(work)),
    }


case_dirs = find_case_dirs()
print(f"Found {len(case_dirs)} case folder(s):")
for p in case_dirs:
    print("  ", p)

Found 0 case folder(s):


## 3. Summary table for all noise values

This table is built from each per-`D` `case_summary.csv`.

In [48]:
summary_rows = []
for case_dir in case_dirs:
    row = read_key_value_csv(case_dir / "case_summary.csv")
    if row:
        row["case_dir"] = case_dir.name
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    for col in summary_df.columns:
        converted = pd.to_numeric(summary_df[col], errors="coerce")
        if converted.notna().sum() == summary_df[col].notna().sum():
            summary_df[col] = converted
    display_cols = [c for c in [
        "case_dir", "D", "n_iter", "n_iter_effective", "n_iter_factor",
        "block_dxi", "local_dxi", "n_prod_obs_total", "n_prod_chains",
        "roundtrip_avg_target_fraction", "roundtrip_convergence_hits",
        "muca_stop_iteration", "muca_stopped_early", "muca_stop_consecutive_hits",
        "muca_stop_target_avg_roundtrips_per_chain",
        "endpoint_x_gt_0p5_n_saved_trajectories", "endpoint_x_gt_0p5_n_transition_trajectories",
        "endpoint_x_gt_0p5_transition_fraction", "endpoint_x_gt_0p5_transition_weight_ess",
        "roundtrip_steps_estimate", "roundtrip_steps_converged",
        "roundtrip_steps_rel_half_range", "reweighting_ess", "reweighting_n_samples"
    ] if c in summary_df.columns]
    if "D" in display_cols:
        display(summary_df[display_cols].sort_values("D", ascending=False))
    else:
        display(summary_df[display_cols])
else:
    print("No case_summary.csv files found. Check RUN_DIR and DATA_DIR.")

No case_summary.csv files found. Check RUN_DIR and DATA_DIR.


## 4. Cross-noise roundtrip scaling plots

These plots use:

- `data/roundtrip_scaling_rows.csv`
- `data/roundtrip_scaling_power_law_fit.csv`

In [49]:
rt_path = DATA_DIR / "roundtrip_scaling_rows.csv"
fit_path = DATA_DIR / "roundtrip_scaling_power_law_fit.csv"
rt_df = read_csv_if_exists(rt_path)
fit_df = read_csv_if_exists(fit_path)

if rt_df is not None and not rt_df.empty:
    for col in ["D", "invD", "steps_estimate", "n_iter_effective", "final_iter_roundtrips", "rel_half_range"]:
        if col in rt_df.columns:
            rt_df[col] = pd.to_numeric(rt_df[col], errors="coerce")
    rt_df = rt_df.sort_values("D", ascending=False)
    display(rt_df)

    valid = finite_df(rt_df, ["invD", "steps_estimate"])
    valid = valid[(valid["invD"] > 0) & (valid["steps_estimate"] > 0)]
    if not valid.empty:
        fig, ax = plt.subplots(figsize=(6.2, 4.2))
        ax.plot(valid["invD"], valid["steps_estimate"], marker="o", linestyle="")
        if fit_df is not None and not fit_df.empty and bool(fit_df.get("valid", pd.Series([False])).iloc[0]):
            C = float(fit_df["C"].iloc[0])
            alpha = float(fit_df["alpha"].iloc[0])
            xs = np.geomspace(valid["invD"].min(), valid["invD"].max(), 200)
            ax.plot(xs, C * xs**alpha, linestyle="--", label=f"fit: C(1/D)^α, α={alpha:.3g}")
            ax.legend()
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("1 / D")
        ax.set_ylabel("Estimated MCMC moves per target roundtrips")
        ax.set_title("Roundtrip cost scaling")
        ax.grid(True, which="both", alpha=0.3)
        save_and_maybe_show(fig, "roundtrip_cost_scaling.png")

    if {"D", "n_iter_effective"}.issubset(rt_df.columns):
        valid = finite_df(rt_df, ["D", "n_iter_effective"])
        if not valid.empty:
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(valid["D"], valid["n_iter_effective"], marker="o")
            ax.set_xscale("log")
            ax.set_xlabel("D")
            ax.set_ylabel("Effective MUCA iterations")
            ax.set_title("Noise-dependent MUCA iteration schedule")
            ax.grid(True, which="both", alpha=0.3)
            save_and_maybe_show(fig, "effective_iterations_vs_D.png")

    if {"D", "final_iter_roundtrips"}.issubset(rt_df.columns):
        valid = finite_df(rt_df, ["D", "final_iter_roundtrips"])
        if not valid.empty:
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(valid["D"], valid["final_iter_roundtrips"], marker="o")
            ax.set_xscale("log")
            ax.set_xlabel("D")
            ax.set_ylabel("Roundtrips in final MUCA iteration")
            ax.set_title("Final-iteration roundtrips by noise level")
            ax.grid(True, which="both", alpha=0.3)
            save_and_maybe_show(fig, "final_roundtrips_vs_D.png")
else:
    print("No cross-noise roundtrip scaling CSV found yet.")

Missing, skipped: C:\Users\amarin\Desktop\ABP_Rare_Events\ABPNoiseSweep\abp_noise_sweep_smoke_test\data\roundtrip_scaling_rows.csv
Missing, skipped: C:\Users\amarin\Desktop\ABP_Rare_Events\ABPNoiseSweep\abp_noise_sweep_smoke_test\data\roundtrip_scaling_power_law_fit.csv
No cross-noise roundtrip scaling CSV found yet.


## 5. MUCA diagnostics for each noise value

For each case folder, this section plots:

- acceptance versus MUCA iteration
- flatness diagnostics versus MUCA iteration
- roundtrips versus MUCA iteration
- roundtrip rate versus MUCA iteration
- estimated steps per target roundtrips versus MUCA iteration
- final MUCA histogram/logweights

In [50]:
for case_dir in case_dirs:
    label = label_for_case(case_dir)
    tag = case_tag(case_dir)
    diag = read_csv_if_exists(case_dir / "muca_iteration_diagnostics.csv")
    if diag is not None and not diag.empty:
        for col in diag.columns:
            diag[col] = pd.to_numeric(diag[col], errors="coerce")

        if {"iteration", "acceptance"}.issubset(diag.columns):
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(diag["iteration"], diag["acceptance"], marker=".")
            ax.set_xlabel("MUCA iteration")
            ax.set_ylabel("Acceptance")
            ax.set_title(f"MUCA acceptance, {label}")
            ax.grid(True, alpha=0.3)
            save_and_maybe_show(fig, f"{tag}_muca_acceptance.png")

        if {"iteration", "flatness_max_over_mean", "flatness_mean_over_min"}.issubset(diag.columns):
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(diag["iteration"], diag["flatness_max_over_mean"], marker=".", label="max / mean")
            ax.plot(diag["iteration"], diag["flatness_mean_over_min"], marker=".", label="mean / min")
            ax.set_yscale("log")
            ax.set_xlabel("MUCA iteration")
            ax.set_ylabel("Flatness ratio")
            ax.set_title(f"MUCA flatness diagnostics, {label}")
            ax.grid(True, which="both", alpha=0.3)
            ax.legend()
            save_and_maybe_show(fig, f"{tag}_muca_flatness.png")

        if {"iteration", "roundtrips"}.issubset(diag.columns):
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(diag["iteration"], diag["roundtrips"], marker=".")
            ax.set_xlabel("MUCA iteration")
            ax.set_ylabel("Roundtrips")
            ax.set_title(f"Roundtrips per MUCA iteration, {label}")
            ax.grid(True, alpha=0.3)
            save_and_maybe_show(fig, f"{tag}_muca_roundtrips.png")

        if {"iteration", "roundtrips_per_sampling_step"}.issubset(diag.columns):
            valid = diag[np.isfinite(diag["roundtrips_per_sampling_step"]) & (diag["roundtrips_per_sampling_step"] > 0)]
            if not valid.empty:
                fig, ax = plt.subplots(figsize=(6.2, 4.2))
                ax.plot(valid["iteration"], valid["roundtrips_per_sampling_step"], marker=".")
                ax.set_yscale("log")
                ax.set_xlabel("MUCA iteration")
                ax.set_ylabel("Roundtrips per sampling step")
                ax.set_title(f"Roundtrip rate, {label}")
                ax.grid(True, which="both", alpha=0.3)
                save_and_maybe_show(fig, f"{tag}_muca_roundtrip_rate.png")

        if {"iteration", "steps_per_target_roundtrips"}.issubset(diag.columns):
            valid = diag[np.isfinite(diag["steps_per_target_roundtrips"]) & (diag["steps_per_target_roundtrips"] > 0)]
            if not valid.empty:
                fig, ax = plt.subplots(figsize=(6.2, 4.2))
                ax.plot(valid["iteration"], valid["steps_per_target_roundtrips"], marker=".")
                ax.set_yscale("log")
                ax.set_xlabel("MUCA iteration")
                ax.set_ylabel("Estimated steps per target roundtrips")
                ax.set_title(f"Roundtrip cost estimate, {label}")
                ax.grid(True, which="both", alpha=0.3)
                save_and_maybe_show(fig, f"{tag}_muca_steps_per_target_roundtrips.png")

    muca_last = read_csv_if_exists(case_dir / "muca_last_histogram_logweights.csv")
    if muca_last is not None and not muca_last.empty:
        for col in muca_last.columns:
            muca_last[col] = pd.to_numeric(muca_last[col], errors="coerce")

        if {"x_T_center", "last_hist_count"}.issubset(muca_last.columns):
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(muca_last["x_T_center"], muca_last["last_hist_count"], marker=".")
            ax.set_yscale("symlog", linthresh=1.0)
            ax.set_xlabel("x(T)")
            ax.set_ylabel("Final MUCA histogram count")
            ax.set_title(f"Final MUCA histogram, {label}")
            ax.grid(True, which="both", alpha=0.3)
            save_and_maybe_show(fig, f"{tag}_muca_final_histogram.png")

        if {"x_T_center", "last_logweight"}.issubset(muca_last.columns):
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(muca_last["x_T_center"], muca_last["last_logweight"])
            ax.set_xlabel("x(T)")
            ax.set_ylabel("Final MUCA logweight")
            ax.set_title(f"Final MUCA logweights, {label}")
            ax.grid(True, alpha=0.3)
            save_and_maybe_show(fig, f"{tag}_muca_final_logweights.png")

        if {"x_T_center", "inverse_weight_relative_to_x0"}.issubset(muca_last.columns):
            valid = muca_last[np.isfinite(muca_last["inverse_weight_relative_to_x0"]) & (muca_last["inverse_weight_relative_to_x0"] > 0)]
            if not valid.empty:
                fig, ax = plt.subplots(figsize=(6.2, 4.2))
                ax.plot(valid["x_T_center"], valid["inverse_weight_relative_to_x0"])
                ax.set_yscale("log")
                ax.set_xlabel("x(T)")
                ax.set_ylabel("Inverse weight relative to x0")
                ax.set_title(f"Relative inverse MUCA weight, {label}")
                ax.grid(True, which="both", alpha=0.3)
                save_and_maybe_show(fig, f"{tag}_muca_inverse_weight_relative_to_x0.png")

In [51]:
transition_rows = []
for case_dir in case_dirs:
    row = read_key_value_csv(case_dir / "case_summary.csv")
    if not row:
        continue
    row["case_dir"] = case_dir.name

    traj_df = read_csv_if_exists(case_dir / "endpoint_x_gt_0p5_trajectory_summary.csv")
    if traj_df is not None and not traj_df.empty and "trajectory_id" in traj_df.columns:
        for col in ["trajectory_id", "endpoint_x", "endpoint_y", "unbiased_weight", "point_count", "kept_point_count"]:
            if col in traj_df.columns:
                traj_df[col] = pd.to_numeric(traj_df[col], errors="coerce")
        n_saved = float(len(traj_df))
        n_trans = float(len(traj_df[np.isfinite(traj_df.get("endpoint_x", np.nan)) & (traj_df["endpoint_x"] > 0.5)]))
    else:
        n_saved = pd.to_numeric(pd.Series([row.get("endpoint_x_gt_0p5_n_saved_trajectories", np.nan)]), errors="coerce").iloc[0]
        n_trans = pd.to_numeric(pd.Series([row.get("endpoint_x_gt_0p5_n_transition_trajectories", np.nan)]), errors="coerce").iloc[0]

    row["endpoint_x_gt_0p5_n_saved_trajectories"] = n_saved
    row["endpoint_x_gt_0p5_n_transition_trajectories"] = n_trans
    row["endpoint_x_gt_0p5_transition_rate"] = (n_trans / n_saved) if pd.notna(n_saved) and pd.notna(n_trans) and n_saved > 0 else np.nan
    transition_rows.append(row)

transition_df = pd.DataFrame(transition_rows)
if transition_df.empty:
    print("No transition summary data found.")
else:
    for col in ["D", "endpoint_x_gt_0p5_n_saved_trajectories", "endpoint_x_gt_0p5_n_transition_trajectories", "endpoint_x_gt_0p5_transition_rate"]:
        if col in transition_df.columns:
            transition_df[col] = pd.to_numeric(transition_df[col], errors="coerce")
    show_cols = [c for c in ["case_dir", "D", "endpoint_x_gt_0p5_n_saved_trajectories", "endpoint_x_gt_0p5_n_transition_trajectories", "endpoint_x_gt_0p5_transition_rate"] if c in transition_df.columns]
    display(transition_df[show_cols].sort_values("D", ascending=False))

    valid = finite_df(transition_df, ["D", "endpoint_x_gt_0p5_n_saved_trajectories", "endpoint_x_gt_0p5_n_transition_trajectories", "endpoint_x_gt_0p5_transition_rate"])
    valid = valid[(valid["D"] > 0) & (valid["endpoint_x_gt_0p5_n_saved_trajectories"] > 0)]
    if not valid.empty:
        fig, ax = plt.subplots(figsize=(6.2, 4.2))
        ax.plot(valid["D"], valid["endpoint_x_gt_0p5_n_transition_trajectories"], marker="o")
        ax.set_xscale("log")
        ax.set_xlabel("D")
        ax.set_ylabel("Transition trajectories")
        ax.set_title("Transition count vs noise level")
        ax.grid(True, which="both", alpha=0.3)
        save_and_maybe_show(fig, "transition_counts_vs_D.png")

        fig, ax = plt.subplots(figsize=(6.2, 4.2))
        ax.plot(valid["D"], valid["endpoint_x_gt_0p5_transition_rate"], marker="o")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("D")
        ax.set_ylabel("Transition rate")
        ax.set_title("Transition rate vs noise level")
        ax.grid(True, which="both", alpha=0.3)
        save_and_maybe_show(fig, "transition_rate_vs_D.png")

        fit = fit_transition_law(valid)
        if fit is not None:
            print(f"Fit: log r(D) = log A + Barrier / D")
            print(f"  A = {fit['A']:.6g}")
            print(f"  Barrier = {fit['Barrier']:.6g}")
            xs = np.geomspace(valid["D"].min(), valid["D"].max(), 200)
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(valid["D"], valid["endpoint_x_gt_0p5_transition_rate"], marker="o", linestyle="")
            ax.plot(xs, fit["A"] * np.exp(fit["Barrier"] / xs), linestyle="--", label=f"fit: A exp(Barrier/D)")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel("D")
            ax.set_ylabel("Transition rate")
            ax.set_title("Transition rate fit")
            ax.grid(True, which="both", alpha=0.3)
            ax.legend()
            save_and_maybe_show(fig, "transition_rate_fit.png")

No transition summary data found.


## 8. Transition numbers and noise dependence

This section summarizes how many trajectories transition to x(T)>0.5 at each noise level and fits an Arrhenius-like law for the transition rate.

## 6. Endpoint and endpoint-conditioned path histograms

For each noise value, this section plots:

- endpoint distribution of `x(T)`
- whole-path `y` distribution conditioned on `x(T) > 0.5`, with path points filtered to `x(t)>-0.3`
- whole-path `(x, y)` heatmaps conditioned on `x(T) > 0.5`, with path points filtered to `x(t)>-0.3`

In [52]:
PATH_X_RANGE = (-0.3, None)


def plot_two_pdfs(df, xcol, title, xlabel, filename, x_range=None):
    cols = {xcol, "biased_pdf", "unbiased_pdf"}
    if not cols.issubset(df.columns):
        print(f"Missing columns for {filename}: needed {cols}")
        return None

    work = df.copy()
    for col in [xcol, "biased_pdf", "unbiased_pdf"]:
        work[col] = pd.to_numeric(work[col], errors="coerce")

    work = work.dropna(subset=[xcol, "biased_pdf", "unbiased_pdf"])
    work = restrict_range(work, xcol, x_range)

    if work.empty:
        print(f"No data inside x_range={x_range} for {filename}")
        return None

    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    ax.plot(work[xcol], work["biased_pdf"], label="biased")
    ax.plot(work[xcol], work["unbiased_pdf"], label="unbiased")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("PDF")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()

    return save_and_maybe_show(fig, filename)


def plot_path_trajectories(path_df, title, filename, max_traj=120):
    if path_df is None or path_df.empty:
        print(f"No path ensemble data for {filename}")
        return None
    needed = {"trajectory_id", "time_index", "x", "y", "unbiased_weight"}
    if not needed.issubset(path_df.columns):
        print(f"Missing columns for {filename}: needed {needed}")
        return None
    work = path_df.copy()
    for col in ["trajectory_id", "time_index", "x", "y", "unbiased_weight"]:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    work = work.dropna()
    traj_ids = list(dict.fromkeys(work["trajectory_id"].astype(int).tolist()))[:max_traj]
    if not traj_ids:
        print(f"No trajectories to plot for {filename}")
        return None
    fig, ax = plt.subplots(figsize=(6.8, 5.2))
    for traj_id in traj_ids:
        sub = work[work["trajectory_id"] == traj_id].sort_values("time_index")
        if sub.empty:
            continue
        ax.plot(sub["x"], sub["y"], alpha=0.18, linewidth=0.9, color="#3b7dd8")
    ax.set_xlabel("x(t)")
    ax.set_ylabel("y(t)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    return save_and_maybe_show(fig, filename)


for case_dir in case_dirs:
    label = label_for_case(case_dir)
    tag = case_tag(case_dir)

    traj_df, path_df = load_transition_ensemble(case_dir)
    if traj_df is not None and not traj_df.empty:
        for col in ["trajectory_id", "endpoint_x", "endpoint_y", "unbiased_weight", "point_count", "kept_point_count"]:
            if col in traj_df.columns:
                traj_df[col] = pd.to_numeric(traj_df[col], errors="coerce")
        if {"trajectory_id", "endpoint_x", "endpoint_y"}.issubset(traj_df.columns):
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.scatter(traj_df["endpoint_x"], traj_df["endpoint_y"], s=22, c=traj_df.get("unbiased_weight", 1.0), cmap="viridis", alpha=0.8)
            ax.axvline(0.5, color="k", linestyle="--", linewidth=1.0)
            ax.set_xlabel("x(T)")
            ax.set_ylabel("y(T)")
            ax.set_title(f"Trajectory endpoints with x(T)>0.5, {label}")
            ax.grid(True, alpha=0.25)
            save_and_maybe_show(fig, f"{tag}_endpoint_x_gt_0p5_endpoints.png")

            kept = traj_df[np.isfinite(traj_df["kept_point_count"]) & (traj_df["kept_point_count"] > 0)]
            if not kept.empty:
                print(f"{label}: {len(kept)} trajectories, {int(kept['kept_point_count'].sum())} kept path points after x(t)>-0.3 cutoff")

    if path_df is not None and not path_df.empty:
        plot_path_trajectories(
            path_df,
            f"Filtered path ensemble P(x, y | x(T)>0.5), x(t)>-0.3, {label}",
            f"{tag}_endpoint_x_gt_0p5_path_spaghetti.png",
        )
        plot_weighted_heatmap(
            path_df,
            "x",
            "y",
            "unbiased_weight",
            f"Unbiased P(x, y | x(T)>0.5), x(t)>-0.3, {label}",
            f"{tag}_endpoint_x_gt_0p5_path_heatmap.png",
            x_range=PATH_X_RANGE,
            y_range=None,
            bins=90,
            log_scale=True,
        )
    else:
        print(f"No endpoint_x_gt_0p5 path ensemble files found for {label}")

## 7. Conditional endpoint-window histograms

Each file named `conditional_*_one_dimensional_histograms.csv` contains histograms for one endpoint window. The observables are usually:

- `y_T`
- `y_mean`
- `y_int`

In [53]:
for case_dir in case_dirs:
    label = label_for_case(case_dir)
    tag = case_tag(case_dir)
    cond_files = sorted(case_dir.glob("conditional_*_one_dimensional_histograms.csv"))
    if not cond_files:
        print(f"No conditional histogram files found for {label}")
        continue

    for path in cond_files:
        df = read_csv_if_exists(path)
        if df is None or df.empty:
            continue
        for col in ["center", "biased_pdf", "unbiased_pdf"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        window_name = path.name.replace("conditional_", "").replace("_one_dimensional_histograms.csv", "")
        if "observable" not in df.columns:
            continue
        for observable, sub in df.groupby("observable"):
            sub = sub.sort_values("center")
            fig, ax = plt.subplots(figsize=(6.2, 4.2))
            ax.plot(sub["center"], sub["biased_pdf"], label="biased")
            ax.plot(sub["center"], sub["unbiased_pdf"], label="unbiased")
            ax.set_xlabel(str(observable))
            ax.set_ylabel("PDF")
            ax.set_title(f"{observable} in endpoint window {window_name}, {label}")
            ax.grid(True, alpha=0.3)
            ax.legend()
            save_and_maybe_show(fig, f"{tag}_conditional_{safe_name(window_name)}_{safe_name(observable)}.png")

## 8. Optional overlay plots across all noise values

These compare the same observable across all `D` values.

In [54]:
# Overlay endpoint x(T) PDFs across D.
fig, ax = plt.subplots(figsize=(6.6, 4.4))
any_data = False
for case_dir in case_dirs:
    df = read_csv_if_exists(case_dir / "hist_endpoint_x_T.csv")
    if df is None or df.empty or not {"x_T_center", "unbiased_pdf"}.issubset(df.columns):
        continue
    for col in ["x_T_center", "unbiased_pdf"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    ax.plot(df["x_T_center"], df["unbiased_pdf"], label=label_for_case(case_dir))
    any_data = True
if any_data:
    ax.set_xlabel("x(T)")
    ax.set_ylabel("Unbiased PDF")
    ax.set_title("Endpoint x(T) distributions across noise levels")
    ax.grid(True, alpha=0.3)
    ax.legend()
    save_and_maybe_show(fig, "overlay_endpoint_xT_unbiased_pdf.png")
else:
    plt.close(fig)
    print("No endpoint x(T) data found for overlay plot.")

# Overlay final MUCA logweights across D.
fig, ax = plt.subplots(figsize=(6.6, 4.4))
any_data = False
for case_dir in case_dirs:
    df = read_csv_if_exists(case_dir / "muca_last_histogram_logweights.csv")
    if df is None or df.empty or not {"x_T_center", "last_logweight"}.issubset(df.columns):
        continue
    for col in ["x_T_center", "last_logweight"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    ax.plot(df["x_T_center"], df["last_logweight"], label=label_for_case(case_dir))
    any_data = True
if any_data:
    ax.set_xlabel("x(T)")
    ax.set_ylabel("Final MUCA logweight")
    ax.set_title("Final MUCA logweights across noise levels")
    ax.grid(True, alpha=0.3)
    ax.legend()
    save_and_maybe_show(fig, "overlay_muca_final_logweights.png")
else:
    plt.close(fig)
    print("No final MUCA logweight data found for overlay plot.")

No endpoint x(T) data found for overlay plot.
No final MUCA logweight data found for overlay plot.


## 9. Figure list

All saved figures are written to `RUN_DIR/plots`.

In [55]:
print(f"Saved {len(FIGURES)} figure(s) to {PLOTS_DIR.resolve()}")
for path in FIGURES:
    print(path)

Saved 0 figure(s) to C:\Users\amarin\Desktop\ABP_Rare_Events\ABPNoiseSweep\abp_noise_sweep_smoke_test\plots
